In [ ]:
import sys
import os

# Go up one level to the main project directory and add it to Python's path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

In [7]:
import warnings

# Ignore all warnings
warnings.filterwarnings("ignore")

In [8]:
import pandas as pd
import optuna
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier
from src.preprocess import preprocess_data

In [9]:
print("Loading and preprocessing training data...")
train_data = pd.read_csv(r'../data/raw/train.csv')
df = preprocess_data(train_data)

X = df.drop('health_condition', axis=1)
y = df['health_condition']

Loading and preprocessing training data...


In [ ]:
print("Starting Optuna Hyperparameter Tuning with Sample Weights...")

X_train_local, X_test_local, y_train_local, y_test_local = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
def objective(trial):
    # 2. Define the hyperparameter search space
    param = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 7),
        'random_state': 42,
        'n_jobs': -1,
        'eval_metric': 'mlogloss'
    }

    model = XGBClassifier(**param)
    
    # 4. Calculate sample weights for the local training data ONLY
    train_weights = compute_sample_weight(
        class_weight='balanced',
        y=y_train_local
    )

    model.fit(X_train_local, y_train_local, sample_weight=train_weights)
    preds = model.predict(X_test_local)
    macro_f1 = f1_score(y_test_local, preds, average='macro')
    return macro_f1
    
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)

print("\n--- Tuning Complete ---")
print(f"Best Macro F1-Score: {study.best_value:.4f}")
print("Best Hyperparameters:")
for key, value in study.best_params.items():
    print(f"    {key}: {value}")

Starting Optuna Hyperparameter Tuning with Sample Weights...


[I 2026-07-29 21:33:39,691] A new study created in memory with name: no-name-15b56245-1e47-43d4-8b59-8166f00c08d4
[I 2026-07-29 21:33:51,734] Trial 0 finished with value: 0.7717673474184167 and parameters: {'n_estimators': 311, 'learning_rate': 0.16004997428355586, 'max_depth': 4, 'subsample': 0.9556708662680662, 'colsample_bytree': 0.8114822411318474, 'min_child_weight': 1}. Best is trial 0 with value: 0.7717673474184167.
[I 2026-07-29 21:34:04,815] Trial 1 finished with value: 0.7531870268995804 and parameters: {'n_estimators': 276, 'learning_rate': 0.032433165064385906, 'max_depth': 6, 'subsample': 0.9821252976843178, 'colsample_bytree': 0.5351161862555176, 'min_child_weight': 6}. Best is trial 0 with value: 0.7717673474184167.
[I 2026-07-29 21:34:19,563] Trial 2 finished with value: 0.7366509938827517 and parameters: {'n_estimators': 420, 'learning_rate': 0.029411367369653146, 'max_depth': 3, 'subsample': 0.85683894618236, 'colsample_bytree': 0.5925307700260394, 'min_child_weight':


--- Tuning Complete ---
Best Macro F1-Score: 0.8868
Best Hyperparameters:
    n_estimators: 485
    learning_rate: 0.19233650728925847
    max_depth: 10
    subsample: 0.6767655133451045
    colsample_bytree: 0.9786605812318665
    min_child_weight: 6


In [12]:
print("Retraining final model on ALL data with Sample Weights...")

# 1. Calculate weights for the entire 100% dataset
full_weights = compute_sample_weight(
    class_weight='balanced',
    y=y
)

# 2. Build the final model unpacking the best Optuna parameters
final_model = XGBClassifier(
    **study.best_params, 
    random_state=42, 
    n_jobs=-1, 
    eval_metric='mlogloss'
)

final_model.fit(X, y, sample_weight=full_weights)

print("Model successfully trained! Ready to process test.csv...")

Retraining final model on ALL data with Sample Weights...
Model successfully trained! Ready to process test.csv...


In [13]:
print("Processing Kaggle test data...")
raw_test_df = pd.read_csv(r'../data/raw/test.csv')
passenger_ids = raw_test_df['id']

clean_test_df = preprocess_data(raw_test_df)
X_test_kaggle = clean_test_df.reindex(columns=X.columns, fill_value=0)

# 4. Generate Predictions
print("Generating final predictions...")
kaggle_preds = final_model.predict(X_test_kaggle)

# 5. Format and Save Submission
submission = pd.DataFrame({
    'id': passenger_ids,
    'health_condition': kaggle_preds
})

# Map numeric predictions back to text labels for Kaggle
reverse_mapping = {0: 'at-risk', 1: 'fit', 2: 'unhealthy'}
submission['health_condition'] = submission['health_condition'].map(reverse_mapping)

submission.to_csv('../results/submission_7.csv', index=False)
print("Success! submission.csv is ready for Kaggle upload.")

Processing Kaggle test data...
Generating final predictions...
Success! submission.csv is ready for Kaggle upload.


## Random Forest

In [18]:
from sklearn.ensemble import RandomForestClassifier

def objective_rf(trial):
    param = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'max_depth': trial.suggest_int('max_depth', 5, 30),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 20),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2']),

        'class_weight': 'balanced',
        'random_state': 42,
        'n_jobs': -1
    }

    rf_model = RandomForestClassifier(**param)
    rf_model.fit(X_train_local, y_train_local)

    preds = rf_model.predict(X_test_local)

    macro_f1 = f1_score(y_test_local, preds, average='macro')
    return macro_f1

study_rf = optuna.create_study(direction='maximize')
study_rf.optimize(objective_rf, n_trials=30)

print(f"Best RF Macro F1: {study_rf.best_value:.4f}")
print("Best RF Params:", study_rf.best_params)

[I 2026-07-29 22:05:33,790] A new study created in memory with name: no-name-95af8b20-9866-4f32-95bc-3e9f7f374aa8
[I 2026-07-29 22:06:05,746] Trial 0 finished with value: 0.8511591419136915 and parameters: {'n_estimators': 261, 'max_depth': 20, 'min_samples_split': 19, 'min_samples_leaf': 12, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.8511591419136915.
[I 2026-07-29 22:06:32,191] Trial 1 finished with value: 0.75107616208083 and parameters: {'n_estimators': 274, 'max_depth': 11, 'min_samples_split': 4, 'min_samples_leaf': 6, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.8511591419136915.
[I 2026-07-29 22:07:06,267] Trial 2 finished with value: 0.8717706496763262 and parameters: {'n_estimators': 297, 'max_depth': 29, 'min_samples_split': 13, 'min_samples_leaf': 17, 'max_features': 'log2'}. Best is trial 2 with value: 0.8717706496763262.
[I 2026-07-29 22:07:57,327] Trial 3 finished with value: 0.7916232564819562 and parameters: {'n_estimators': 463, 'max_depth': 14, 

Best RF Macro F1: 0.9008
Best RF Params: {'n_estimators': 156, 'max_depth': 27, 'min_samples_split': 8, 'min_samples_leaf': 1, 'max_features': 'log2'}


In [ ]:
from sklearn.metrics import classification_report, f1_score

# ==========================================
# PHASE 1: LOCAL VALIDATION (Using Optuna's Best Params)
# ==========================================
print("--- Phase 1: Local Validation ---")

# 1. Split the data to create a local holdout set
X_train_val, X_test_val, y_train_val, y_test_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 2. Build the validation model with Optuna's best params
rf_val_model = RandomForestClassifier(
    **study_rf.best_params, 
    class_weight='balanced',
    random_state=42, 
    n_jobs=-1, 
)

# 3. Train and predict on the local split
print("Training validation model...")
rf_val_model.fit(X_train_val, y_train_val)
val_preds = rf_val_model.predict(X_test_val)

# 4. Print detailed metrics
print("\nLocal Validation Metrics:")
print(classification_report(y_test_val, val_preds))

macro_f1 = f1_score(y_test_val, val_preds, average='macro')
print(f"Validation Macro F1-Score: {macro_f1:.4f}\n")

--- Phase 1: Local Validation ---
Training validation model...

Local Validation Metrics:
              precision    recall  f1-score   support

           0       0.97      0.99      0.98    118512
           1       0.93      0.81      0.86      7961
           2       0.93      0.80      0.86     11545

    accuracy                           0.96    138018
   macro avg       0.94      0.87      0.90    138018
weighted avg       0.96      0.96      0.96    138018

Validation Macro F1-Score: 0.9008



In [ ]:
# 2. Retrain the model on 100% of the SMOTE Data using OPTUNA'S BEST PARAMS
print("Retraining final model with optimized parameters...")

# We unpack **study.best_params to automatically pass in the winning settings
rf_model = RandomForestClassifier(
    **study_rf.best_params, 
    class_weight='balanced',
    random_state=42, 
    n_jobs=-1, 
)

rf_model.fit(X, y)

# 3. Load and preprocess the Kaggle test data
print("Processing Kaggle test data...")
raw_test_df = pd.read_csv(r'../data/raw/test.csv')
passenger_ids = raw_test_df['id']

clean_test_df = preprocess_data(raw_test_df)
X_test_kaggle = clean_test_df.reindex(columns=X.columns, fill_value=0)

# 4. Generate Predictions
print("Generating final predictions...")
kaggle_preds = rf_model.predict(X_test_kaggle)

# 5. Format and Save Submission
submission = pd.DataFrame({
    'id': passenger_ids,
    'health_condition': kaggle_preds
})

# Map numeric predictions back to text labels for Kaggle
reverse_mapping = {0: 'at-risk', 1: 'fit', 2: 'unhealthy'}
submission['health_condition'] = submission['health_condition'].map(reverse_mapping)

submission.to_csv('../results/submission_8.csv', index=False)
print("Success! submission.csv is ready for Kaggle upload.")

Retraining final model with optimized parameters...
Processing Kaggle test data...
Generating final predictions...
Success! submission.csv is ready for Kaggle upload.


## Catboost

In [22]:
from catboost import CatBoostClassifier

def objective_cat(trial):
    param = {
        'iterations': trial.suggest_int('iterations', 100, 1000),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'depth': trial.suggest_int('depth', 4, 10),
        'l2_leaf_reg': trial.suggest_int('l2_leaf_reg', 1, 10),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.0),
        
        # Keep these static for the competition requirements
        'auto_class_weights': 'Balanced',
        'random_state': 42,
        'verbose': 0 # Prevents CatBoost from printing a massive log for every trial
    }
    cat_model = CatBoostClassifier(**param)
    cat_model.fit(X_train_local, y_train_local)
    
    # 3. Predict on the untouched local test set
    preds = cat_model.predict(X_test_local)
    
    # 4. Evaluate using Macro F1
    macro_f1 = f1_score(y_test_local, preds, average='macro')
    return macro_f1

study_cat = optuna.create_study(direction='maximize')
study_cat.optimize(objective_cat, n_trials=30)

print(f"Best CatBoost Macro F1: {study_cat.best_value:.4f}")
print("Best CatBoost Params:", study_cat.best_params)

[I 2026-07-29 22:30:06,496] A new study created in memory with name: no-name-86ac4eae-42de-4e57-9b0d-9791e30fcf9a
[I 2026-07-29 22:30:42,639] Trial 0 finished with value: 0.7614780320921368 and parameters: {'iterations': 795, 'learning_rate': 0.03379759158142514, 'depth': 6, 'l2_leaf_reg': 10, 'bagging_temperature': 0.25460048870980667}. Best is trial 0 with value: 0.7614780320921368.
[I 2026-07-29 22:30:47,590] Trial 1 finished with value: 0.7020443656493743 and parameters: {'iterations': 124, 'learning_rate': 0.028258171025921004, 'depth': 4, 'l2_leaf_reg': 6, 'bagging_temperature': 0.2408258713327014}. Best is trial 0 with value: 0.7614780320921368.
[I 2026-07-29 22:31:57,680] Trial 2 finished with value: 0.7622661170229823 and parameters: {'iterations': 400, 'learning_rate': 0.03442691505534069, 'depth': 10, 'l2_leaf_reg': 8, 'bagging_temperature': 0.3986574394065111}. Best is trial 2 with value: 0.7622661170229823.
[I 2026-07-29 22:32:23,309] Trial 3 finished with value: 0.7701776

Best CatBoost Macro F1: 0.8329
Best CatBoost Params: {'iterations': 882, 'learning_rate': 0.1255177474322984, 'depth': 10, 'l2_leaf_reg': 2, 'bagging_temperature': 0.46269931475793813}


In [23]:
import json

# Bundle all the best parameters into one dictionary
best_params_dict = {
    'random_forest': study_rf.best_params,
    'catboost': study_cat.best_params,
    'xgboost': study.best_params
}

# Save to a JSON file
with open('../results/best_models_params.json', 'w') as file:
    json.dump(best_params_dict, file, indent=4)

print("Best parameters successfully saved to best_models_params.json!")

Best parameters successfully saved to best_models_params.json!


In [24]:
print("--- Phase 1: Local Validation ---")

# 1. Split the data to create a local holdout set
X_train_val, X_test_val, y_train_val, y_test_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 2. Build the validation model with Optuna's best params
cat_model = CatBoostClassifier(
    **study_cat.best_params,
    auto_class_weights='Balanced',
    random_state=42,
    verbose=0 # Keeps your notebook clean
)

# 3. Train and predict on the local split
print("Training validation model...")
cat_model.fit(X_train_val, y_train_val)
val_preds = cat_model.predict(X_test_val)

# 4. Print detailed metrics
print("\nLocal Validation Metrics:")
print(classification_report(y_test_val, val_preds))

macro_f1 = f1_score(y_test_val, val_preds, average='macro')
print(f"Validation Macro F1-Score: {macro_f1:.4f}\n")

--- Phase 1: Local Validation ---
Training validation model...

Local Validation Metrics:
              precision    recall  f1-score   support

           0       0.98      0.93      0.96    118512
           1       0.68      0.87      0.76      7961
           2       0.69      0.89      0.78     11545

    accuracy                           0.93    138018
   macro avg       0.78      0.90      0.83    138018
weighted avg       0.94      0.93      0.93    138018

Validation Macro F1-Score: 0.8329



In [26]:
# 2. Retrain the model on 100% of the SMOTE Data using OPTUNA'S BEST PARAMS
print("Retraining final model with optimized parameters...")

# We unpack **study.best_params to automatically pass in the winning settings
cat_model = CatBoostClassifier(
    **study_cat.best_params,
    auto_class_weights='Balanced',
    random_state=42,
    verbose=0 # Keeps your notebook clean
)


cat_model.fit(X, y)

# 3. Load and preprocess the Kaggle test data
print("Processing Kaggle test data...")
raw_test_df = pd.read_csv(r'../data/raw/test.csv')
passenger_ids = raw_test_df['id']

clean_test_df = preprocess_data(raw_test_df)
X_test_kaggle = clean_test_df.reindex(columns=X.columns, fill_value=0)

# 4. Generate Predictions
print("Generating final predictions...")
kaggle_preds = cat_model.predict(X_test_kaggle)
kaggle_preds = kaggle_preds.flatten()
# 5. Format and Save Submission
submission = pd.DataFrame({
    'id': passenger_ids,
    'health_condition': kaggle_preds
})

# Map numeric predictions back to text labels for Kaggle
reverse_mapping = {0: 'at-risk', 1: 'fit', 2: 'unhealthy'}
submission['health_condition'] = submission['health_condition'].map(reverse_mapping)

submission.to_csv('../results/submission_9.csv', index=False)
print("Success! submission.csv is ready for Kaggle upload.")

Retraining final model with optimized parameters...
Processing Kaggle test data...
Generating final predictions...
Success! submission.csv is ready for Kaggle upload.


## Voting Classifier

In [ ]:
from sklearn.ensemble import VotingClassifier
from xgboost import XGBClassifier

# ==========================================
# PHASE 1: LOCAL VALIDATION (Ensemble)
# ==========================================
print("--- Phase 1: Local Validation for Voting Classifier ---")

X_train_val, X_test_val, y_train_val, y_test_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

rf_best = RandomForestClassifier(
    **study_rf.best_params, 
    class_weight='balanced',
    random_state=42, 
    n_jobs=-1
)

cat_best = CatBoostClassifier(
    **study_cat.best_params,
    auto_class_weights='Balanced',
    random_state=42,
    verbose=0
)

# Assuming your XGBoost study is named study_xgb
xgb_best = XGBClassifier(
    **study.best_params,
    random_state=42,
    n_jobs=-1
)


voting_val_model = VotingClassifier(
    estimators=[
        ('rf', rf_best),
        ('cat', cat_best),
        ('xgb', xgb_best)
    ],
    voting='soft'
)

print("Training Voting Classifier...")
voting_val_model.fit(X_train_val, y_train_val)
val_preds = voting_val_model.predict(X_test_val)

print("\nEnsemble Local Validation Metrics:")
print(classification_report(y_test_val, val_preds))
print(f"Validation Macro F1-Score: {f1_score(y_test_val, val_preds, average='macro'):.4f}\n")

--- Phase 1: Local Validation for Voting Classifier ---
Training Voting Classifier...

Ensemble Local Validation Metrics:
              precision    recall  f1-score   support

           0       0.97      0.99      0.98    118512
           1       0.91      0.82      0.86      7961
           2       0.90      0.81      0.86     11545

    accuracy                           0.96    138018
   macro avg       0.93      0.87      0.90    138018
weighted avg       0.96      0.96      0.96    138018

Validation Macro F1-Score: 0.8990



In [28]:
# ==========================================
# PHASE 2: FINAL RETRAINING & KAGGLE SUBMISSION
# ==========================================
print("--- Phase 2: Full Retraining & Kaggle Submission ---")

# 1. Rebuild the final ensemble for the full dataset
voting_final_model = VotingClassifier(
    estimators=[
        ('rf', rf_best), 
        ('cat', cat_best), 
        ('xgb', xgb_best)
    ],
    voting='soft'
)

print("Retraining Voting Classifier on ALL data...")
voting_final_model.fit(X, y)

# 2. Process Kaggle test data
print("Processing Kaggle test data...")
raw_test_df = pd.read_csv(r'../data/raw/test.csv')
passenger_ids = raw_test_df['id']

clean_test_df = preprocess_data(raw_test_df)
X_test_kaggle = clean_test_df.reindex(columns=X.columns, fill_value=0)

# 3. Generate Predictions
print("Generating final ensemble predictions...")
kaggle_preds = voting_final_model.predict(X_test_kaggle)

# 4. Format and Save Submission
submission = pd.DataFrame({
    'id': passenger_ids,
    'health_condition': kaggle_preds
})

reverse_mapping = {0: 'at-risk', 1: 'fit', 2: 'unhealthy'}
submission['health_condition'] = submission['health_condition'].map(reverse_mapping)

submission.to_csv('../results/submission_10.csv', index=False)
print("Success! submission_ensemble.csv is ready for Kaggle upload.")

--- Phase 2: Full Retraining & Kaggle Submission ---
Retraining Voting Classifier on ALL data...
Processing Kaggle test data...
Generating final ensemble predictions...
Success! submission_ensemble.csv is ready for Kaggle upload.


### Voting Classifier with sample weights

In [30]:
from sklearn.ensemble import VotingClassifier
from xgboost import XGBClassifier

# ==========================================
# PHASE 1: LOCAL VALIDATION (Ensemble)
# ==========================================
print("--- Phase 1: Local Validation for Voting Classifier ---")

X_train_val, X_test_val, y_train_val, y_test_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
train_weights = compute_sample_weight(
    class_weight='balanced',
    y=y_train_val  # Updated to match the validation split target
)
rf_best = RandomForestClassifier(
    **study_rf.best_params, 
    random_state=42, 
    n_jobs=-1
)

cat_best = CatBoostClassifier(
    **study_cat.best_params,
    random_state=42,
    verbose=0
)

# Assuming your XGBoost study is named study_xgb
xgb_best = XGBClassifier(
    **study.best_params,
    random_state=42,
    n_jobs=-1
)


voting_val_model = VotingClassifier(
    estimators=[
        ('rf', rf_best),
        ('cat', cat_best),
        ('xgb', xgb_best)
    ],
    voting='soft'
)

print("Training Voting Classifier...")
voting_val_model.fit(X_train_val, y_train_val, sample_weight=train_weights)
val_preds = voting_val_model.predict(X_test_val)

print("\nEnsemble Local Validation Metrics:")
print(classification_report(y_test_val, val_preds))
print(f"Validation Macro F1-Score: {f1_score(y_test_val, val_preds, average='macro'):.4f}\n")

--- Phase 1: Local Validation for Voting Classifier ---
Training Voting Classifier...

Ensemble Local Validation Metrics:
              precision    recall  f1-score   support

           0       0.97      0.98      0.98    118512
           1       0.87      0.83      0.85      7961
           2       0.85      0.84      0.84     11545

    accuracy                           0.96    138018
   macro avg       0.90      0.88      0.89    138018
weighted avg       0.96      0.96      0.96    138018

Validation Macro F1-Score: 0.8896



In [31]:
# ==========================================
# PHASE 2: FINAL RETRAINING & KAGGLE SUBMISSION
# ==========================================
print("--- Phase 2: Full Retraining & Kaggle Submission ---")
full_train_weights = compute_sample_weight(class_weight='balanced', y=y)
# 1. Rebuild the final ensemble for the full dataset
voting_final_model = VotingClassifier(
    estimators=[
        ('rf', rf_best), 
        ('cat', cat_best), 
        ('xgb', xgb_best)
    ],
    voting='soft'
)

print("Retraining Voting Classifier on ALL data...")
voting_final_model.fit(X, y, sample_weight=full_train_weights)

# 2. Process Kaggle test data
print("Processing Kaggle test data...")
raw_test_df = pd.read_csv(r'../data/raw/test.csv')
passenger_ids = raw_test_df['id']

clean_test_df = preprocess_data(raw_test_df)
X_test_kaggle = clean_test_df.reindex(columns=X.columns, fill_value=0)

# 3. Generate Predictions
print("Generating final ensemble predictions...")
kaggle_preds = voting_final_model.predict(X_test_kaggle)

# 4. Format and Save Submission
submission = pd.DataFrame({
    'id': passenger_ids,
    'health_condition': kaggle_preds
})

reverse_mapping = {0: 'at-risk', 1: 'fit', 2: 'unhealthy'}
submission['health_condition'] = submission['health_condition'].map(reverse_mapping)

submission.to_csv('../results/submission_11.csv', index=False)
print("Success! submission_ensemble.csv is ready for Kaggle upload.")

--- Phase 2: Full Retraining & Kaggle Submission ---
Retraining Voting Classifier on ALL data...
Processing Kaggle test data...
Generating final ensemble predictions...
Success! submission_ensemble.csv is ready for Kaggle upload.


### submission_3.csv best result  
### submission_6.csv second best result